> Saved output note: repetitive vLLM/transformers log lines (per-layer bf16->fp16 weight-cast warnings, CUDA-graph and progress bars) were trimmed from the saved outputs. All benchmark results, the T4 UUID line, KV-cache sizing and the own-prompt generation are unchanged.

# HW4.5 Part D — Serving revisited (vLLM vs HF pipeline)
Run on Colab with **Runtime > Change runtime type > GPU** (T4, L4 or A100). Run all cells top to bottom (~15 min).
Results land in `partD_results.json` and a printed markdown table to paste into METRICS.md.

Definitions used (state them in your write-up):
- **Request latency** = wall time for a request to complete. For vLLM batch B, B requests are submitted together via `llm.generate`, so per-request latency is the batch completion time (repeated N times, p50/p95 over repetitions).
- **Tokens/sec** = generated tokens summed over the batch / batch wall time.
- Every request generates exactly 128 new tokens (EOS ignored) so configurations are comparable.

In [1]:
!pip -q install vllm
import subprocess; print(subprocess.run(['nvidia-smi','-L'],capture_output=True,text=True).stdout)  # UUID capture for the deliverable

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
numba-cuda 0.28.2 requires cuda-core<1.0.0,>=0.5.1, but you have cuda-core 1.2.2 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
cudf-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
cuml-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
GPU 0: Tesla T4 (UUID: GPU-6cb18f34-3c2f-33e8-fdfb-31b18ae8a8df)


In [2]:
import os, time, json, gc, statistics
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"   # keep the engine in-process so we can read cache_config
import torch
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # change to try another Hugging Face model (assignment asks for a different model + your own prompt)
MY_PROMPT = "Explain in three sentences why GPU memory, not compute, usually limits LLM batch size."  # replace with your own prompt
NEW_TOKENS, REPS = 128, 20
DTYPE = "float16"                      # T4 has no bf16; fine on all Colab GPUs
GPU = torch.cuda.get_device_name(0); TOTAL_GB = torch.cuda.get_device_properties(0).total_memory/2**30
print(GPU, round(TOTAL_GB,1), "GB")
prompts_pool = [MY_PROMPT] + [f"Write a short paragraph about topic number {i}: the history of computing." for i in range(1, 64)]
def pct(xs, p):
    xs = sorted(xs); k = (len(xs)-1)*p/100; f = int(k); c = min(f+1, len(xs)-1)
    return xs[f] + (xs[c]-xs[f])*(k-f)
results = {"gpu": GPU, "model": MODEL, "new_tokens": NEW_TOKENS, "reps": REPS}

Tesla T4 14.6 GB


## 1. HF pipeline baseline, batch 1

In [3]:
from transformers import pipeline, AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL)
pipe = pipeline("text-generation", model=MODEL, device=0, torch_dtype=torch.float16)
gen = dict(max_new_tokens=NEW_TOKENS, min_new_tokens=NEW_TOKENS, do_sample=False, return_full_text=False)
pipe(prompts_pool[0], **gen)  # warmup
lat, toks = [], []
for i in range(REPS):
    t = time.time(); out = pipe(prompts_pool[i % len(prompts_pool)], **gen)[0]["generated_text"]; torch.cuda.synchronize()
    lat.append(time.time()-t); toks.append(len(tok(out).input_ids))
results["hf_b1"] = dict(tok_s=sum(toks)/sum(lat), p50_ms=pct(lat,50)*1e3, p95_ms=pct(lat,95)*1e3)
print(results["hf_b1"]); print("sample output:", out[:200])
del pipe; gc.collect(); torch.cuda.empty_cache()

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'min_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


{'tok_s': 22.487123958025652, 'p50_ms': 5482.2152853012085, 'p95_ms': 6634.914171695711}
sample output:  The history of computing is a fascinating journey that spans thousands of years and has seen significant advancements in technology, from ancient counting systems to modern computers. One of the earl


## 2. vLLM at batch sizes 1, 8, 32

In [4]:
from vllm import LLM, SamplingParams
llm = LLM(MODEL, dtype=DTYPE, gpu_memory_utilization=0.85, max_model_len=2048)
sp = SamplingParams(max_tokens=NEW_TOKENS, min_tokens=NEW_TOKENS, ignore_eos=True, temperature=0.0)
llm.generate(prompts_pool[:2], sp)  # warmup
for B in (1, 8, 32):
    lat, ntok = [], 0
    for r in range(REPS):
        ps = [prompts_pool[(r*B+j) % len(prompts_pool)] for j in range(B)]
        t = time.time(); outs = llm.generate(ps, sp, use_tqdm=False); dt = time.time()-t
        lat.append(dt); ntok += sum(len(o.outputs[0].token_ids) for o in outs)
    results[f"vllm_b{B}"] = dict(tok_s=ntok/sum(lat), p50_ms=pct(lat,50)*1e3, p95_ms=pct(lat,95)*1e3)
    print(B, results[f"vllm_b{B}"])
print(llm.generate([MY_PROMPT], sp)[0].outputs[0].text)  # your own prompt

(MainProcess pid=817) INFO 10-30 23:07:34 [api_utils.py:286] non-default args: {'dtype': 'float16', 'max_model_len': 2048, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}
(MainProcess pid=817) WARNING 10-30 23:07:34 [arg_utils.py:1945] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
(MainProcess pid=817) INFO 10-30 23:07:56 [model.py:733] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=817) WARNING 10-30 23:07:56 [model.py:2498] Casting torch.bfloat16 to torch.float16.
(MainProcess pid=817) INFO 10-30 23:07:56 [model.py:2152] Using max model len 2048
(MainProcess pid=817) INFO 10-30 23:07:56 [scheduler.py:307] Chunked prefill is enabled with max_num_batched_tokens=8192.
(MainProcess pid=817) INFO 10-30 23:07:57 [kernel.py:467] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['nat

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

(MainProcess pid=817) INFO 10-30 23:08:00 [core.py:133] Initializing a V1 LLM engine (v0.32.0) with config: model='Qwen/Qwen2.5-1.5B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-1.5B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=main, tokenizer_revision=main, trust_remote_code=False, dtype=torch.float16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=False, aux_output_config=AuxOutputConfig(enable_return_routed_experts=False, max_bytes=None), kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityCon

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


(MainProcess pid=817) INFO 10-30 23:08:21 [default_loader.py:501] Loading weights took 12.74 seconds
(MainProcess pid=817) INFO 10-30 23:08:23 [model_runner.py:424] Model loading took 2.98 GiB memory and 19.329986 seconds
(MainProcess pid=817) WARNING 10-30 23:08:23 [topk_topp_sampler.py:131] FlashInfer top-p/top-k sampling unavailable: unsupported compute capability 7.5; falling back. Set VLLM_USE_FLASHINFER_SAMPLER=0 to silence.
(MainProcess pid=817) INFO 10-30 23:08:23 [utils.py:347] Using LBNHC KV cache layout.
(MainProcess pid=817) INFO 10-30 23:08:27 [monitor.py:40] [backbone] Starting torch.compile
(MainProcess pid=817) INFO 10-30 23:08:34 [backends.py:1104] [backbone] Using cache directory: /root/.cache/vllm/torch_compile_cache/fdd7859a86/rank_0_0/backbone for vLLM's torch.compile
(MainProcess pid=817) INFO 10-30 23:08:34 [backends.py:1166] [backbone] Dynamo bytecode transform time: 7.85 s


... (4 outputs left out)

Rendering prompts:   0%|          | 0/2 [00:00<?, ?it/s]

(MainProcess pid=817) WARNING 10-30 23:09:43 [jit_monitor.py:140] Triton kernel JIT compilation during inference: kernel_unified_attention. This causes a latency spike; consider extending warmup to cover this shape/config.
1 {'tok_s': 65.9015860722236, 'p50_ms': 1893.188714981079, 'p95_ms': 1981.0285091400153}
8 {'tok_s': 501.90853441369956, 'p50_ms': 2041.558861732483, 'p95_ms': 2044.9964761734009}
32 {'tok_s': 1399.4641538029332, 'p50_ms': 2928.083300590515, 'p95_ms': 2952.002441883087}


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

 GPU memory is limited by the amount of VRAM available, which is typically measured in gigabytes. This means that the amount of data that can be stored in GPU memory is limited, and as a result, the amount of data that can be processed in parallel is also limited. This is why GPU memory, not compute, usually limits the batch size of large language models (LLMs). The amount of data that can be processed in parallel is limited by the amount of VRAM available, which is typically measured in gigabytes. This means that the amount of data that can be stored in GPU memory is limited, and as a result,


## 3. KV-cache memory vs context length (batch 8)
KV bytes/token = 2 (K and V) x layers x kv_heads x head_dim x bytes/elt. We (a) compute it analytically for several context lengths at batch 8 and
(b) read how many KV tokens vLLM could actually allocate on this card, then (c) verify by serving batch 8 at the longest context that fits.

In [5]:
cfg = llm.llm_engine.model_config.hf_config
L, H = cfg.num_hidden_layers, cfg.num_key_value_heads
HD = getattr(cfg, "head_dim", cfg.hidden_size // cfg.num_attention_heads)
per_tok = 2*L*H*HD*2   # fp16 bytes
print("KV bytes/token:", per_tok)
BATCH = 8
table = {ctx: BATCH*ctx*per_tok/2**30 for ctx in (256, 512, 1024, 2048, 4096, 8192, 16384, 32768)}
for c, g in table.items(): print(f"ctx {c:6d} x batch {BATCH}: KV {g:6.2f} GB")
results["kv_gb_vs_ctx_b8"] = table
cc = getattr(llm.llm_engine, "cache_config", None) or llm.llm_engine.vllm_config.cache_config
cap_tokens = cc.num_gpu_blocks * cc.block_size
results["kv_capacity_tokens"] = cap_tokens
results["max_ctx_b8_by_capacity"] = cap_tokens // BATCH
print("KV capacity tokens:", cap_tokens, "-> longest ctx at batch 8:", cap_tokens // BATCH, "(limited also by model max positions:", cfg.max_position_embeddings, ")")

KV bytes/token: 28672
ctx    256 x batch 8: KV   0.05 GB
ctx    512 x batch 8: KV   0.11 GB
ctx   1024 x batch 8: KV   0.22 GB
ctx   2048 x batch 8: KV   0.44 GB
ctx   4096 x batch 8: KV   0.88 GB
ctx   8192 x batch 8: KV   1.75 GB
ctx  16384 x batch 8: KV   3.50 GB
ctx  32768 x batch 8: KV   7.00 GB
KV capacity tokens: 312672 -> longest ctx at batch 8: 39084 (limited also by model max positions: 32768 )


In [6]:
# Verify: re-create the engine with max_model_len = capacity//8 (capped by the model's max positions) and run batch 8 at that length.
del llm; gc.collect(); torch.cuda.empty_cache()
target = min(results["max_ctx_b8_by_capacity"], cfg.max_position_embeddings) - NEW_TOKENS - 8
llm = LLM(MODEL, dtype=DTYPE, gpu_memory_utilization=0.85, max_model_len=target + NEW_TOKENS + 8)
ids = [[1000 + (i % 500)] * target for i in range(8)]
try:
    outs = llm.generate([{"prompt_token_ids": x} for x in ids], SamplingParams(max_tokens=NEW_TOKENS, ignore_eos=True), use_tqdm=False)
    results["verified_ctx_b8"] = target + NEW_TOKENS; print("OK at context", target + NEW_TOKENS)
except Exception as e:
    results["verified_ctx_b8"] = f"FAILED: {type(e).__name__}"; print(results["verified_ctx_b8"])

(MainProcess pid=817) INFO 10-30 23:12:05 [api_utils.py:286] non-default args: {'dtype': 'float16', 'max_model_len': 32768, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}
(MainProcess pid=817) WARNING 10-30 23:12:05 [arg_utils.py:1945] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
(MainProcess pid=817) INFO 10-30 23:12:05 [model.py:733] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=817) WARNING 10-30 23:12:05 [model.py:2498] Casting torch.bfloat16 to torch.float16.
(MainProcess pid=817) INFO 10-30 23:12:05 [model.py:2152] Using max model len 32768
(MainProcess pid=817) INFO 10-30 23:12:05 [kernel.py:467] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])
(MainProcess pid=817) INFO 10-30 23:12:0

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


(MainProcess pid=817) INFO 10-30 23:12:17 [default_loader.py:501] Loading weights took 7.39 seconds
(MainProcess pid=817) INFO 10-30 23:12:20 [model_runner.py:424] Model loading took 2.98 GiB memory and 11.144382 seconds
(MainProcess pid=817) INFO 10-30 23:12:21 [monitor.py:40] [backbone] Starting torch.compile
(MainProcess pid=817) INFO 10-30 23:12:29 [backends.py:1104] [backbone] Using cache directory: /root/.cache/vllm/torch_compile_cache/a7c0052d84/rank_0_0/backbone for vLLM's torch.compile
(MainProcess pid=817) INFO 10-30 23:12:29 [backends.py:1166] [backbone] Dynamo bytecode transform time: 7.76 s
(MainProcess pid=817) INFO 10-30 23:12:35 [backends.py:398] [backbone] Compiling a graph for compile range (1, 8192) takes 5.61 s
(MainProcess pid=817) INFO 10-30 23:12:39 [backends.py:927] collected artifacts: 29 entries, 3 artifacts, 4739102 bytes total
(MainProcess pid=817) INFO 10-30 23:12:40 [decorators.py:743] [backbone] saved AOT compiled function to /root/.cache/vllm/torch_compi

(MainProcess pid=817) INFO 10-30 23:12:48 [model_runner.py:1127] Graph capturing finished in 6 secs, took 0.21 GiB
(MainProcess pid=817) INFO 10-30 23:12:49 [gpu_worker.py:759] Available KV cache memory: 8.45 GiB
(MainProcess pid=817) INFO 10-30 23:12:49 [gpu_worker.py:774] CUDA graph memory profiling is enabled (default since v0.21.0). The current --gpu-memory-utilization=0.8500 is equivalent to --gpu-memory-utilization=0.8316 without CUDA graph memory profiling. To maintain the same effective KV cache size as before, increase --gpu-memory-utilization to 0.8684. To disable, set VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS=0.
(MainProcess pid=817) INFO 10-30 23:12:49 [kv_cache_utils.py:2327] GPU KV cache size: 316,400 tokens, Maximum concurrency for 32,768 tokens per request: 9.66x
(MainProcess pid=817) INFO 10-30 23:12:49 [kernel_warmup.py:176] JIT kernel warmup starting.
(MainProcess pid=817) INFO 10-30 23:12:49 [kernel_warmup.py:189] JIT kernel warmup finished in 0.00s.


(MainProcess pid=817) INFO 10-30 23:12:57 [model_runner.py:1127] Graph capturing finished in 7 secs, took 0.06 GiB
(MainProcess pid=817) INFO 10-30 23:12:57 [gpu_worker.py:971] CUDA graph pool memory: 0.06 GiB (actual), 0.27 GiB (estimated), difference: 0.2 GiB (315.2%).
(MainProcess pid=817) INFO 10-30 23:12:57 [gpu_worker.py:1034] Free memory on device (14.04/14.56 GiB) on startup. Desired GPU memory utilization is (0.85, 12.38 GiB). Actual usage is 3.17 GiB for consumed memory (weights + non-torch), 0.76 GiB for peak activation, and 0.06 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=8845555405` (8.24 GiB) to fit into requested memory, or `--kv-cache-memory=10630261248` (9.9 GiB) to fully utilize gpu memory. Current kv cache memory in use is 8.45 GiB.
(MainProcess pid=817) INFO 10-30 23:12:58 [core.py:395] init engine (profile, create kv cache, warmup model) took 37.34 s (compilation: 18.41 s)
OK at context 32760


## 4. Results table and save

In [7]:
rows = [("HF pipeline, batch 1","hf_b1"),("vLLM, batch 1","vllm_b1"),("vLLM, batch 8","vllm_b8"),("vLLM, batch 32","vllm_b32")]
print("| Configuration | Tokens/sec | p50 latency (ms) | p95 latency (ms) |\n|---|---|---|---|")
for n,k in rows: r=results[k]; print(f"| {n} | {r['tok_s']:.1f} | {r['p50_ms']:.0f} | {r['p95_ms']:.0f} |")
json.dump(results, open("partD_results.json","w"), indent=1)
from google.colab import files; files.download("partD_results.json")

| Configuration | Tokens/sec | p50 latency (ms) | p95 latency (ms) |
|---|---|---|---|
| HF pipeline, batch 1 | 22.5 | 5482 | 6635 |
| vLLM, batch 1 | 65.9 | 1893 | 1981 |
| vLLM, batch 8 | 501.9 | 2042 | 2045 |
| vLLM, batch 32 | 1399.5 | 2928 | 2952 |


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
MY_PROMPT = "Explain in simple terms why increasing GPU batch size can improve training throughput at first, but eventually gives smaller speed gains and can run out of GPU memory. Include how GPU utilization and memory limits contribute to this behavior."
sp_own = SamplingParams(max_tokens=400, temperature=0.0)
out = llm.generate([MY_PROMPT], sp_own)[0].outputs[0].text
print(out)
results["my_prompt"] = MY_PROMPT
results["my_prompt_output"] = out
import json; json.dump(results, open("partD_results.json","w"), indent=1)
from google.colab import files; files.download("partD_results.json")

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

 Increasing the batch size in a machine learning model training process can initially improve training throughput because it allows the model to process more data in a single pass through the training loop. This means that the model can make more updates to its parameters with each iteration, which can lead to faster convergence and improved performance. However, as the batch size increases, the amount of memory required by the model also increases. This is because each additional data point requires more memory to store and process. Eventually, the GPU may run out of memory, which can cause the training process to slow down or even crash. Additionally, as the batch size increases, the amount of time it takes to train the model also increases. This is because the model needs more time to process each batch of data, which can lead to longer training times and slower overall performance. Overall, increasing the batch size can improve training throughput at first, but eventually gives sma

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## For METRICS.md
- Paste the table above, the `nvidia-smi -L` line from cell 1, the KV table and the longest context at batch 8.
- Compare with your **HW4.a** serving numbers (not available to the author of this notebook). If they differ a lot, explain which of: hardware (T4/L4 vs your HW4 GPU), precision (this notebook uses fp16), or a measurement flaw (e.g. warmup excluded here, per-request vs per-batch latency definition).
- Answer the HW4.1 question: batch size where per-request latency exceeds the HF baseline while total throughput still improves; for interactive apps p95 per-request latency matters, not total throughput.